In [148]:
import random

In [149]:
### Init
def init_population(
        number_of_cities: int,
        population_size: int
):
    base = [i for i in range(1, number_of_cities)]
    init_pop = [random.sample(base, len(base)) for _ in range(population_size)]

    return init_pop

In [150]:
### Fitness
def cal_fitness(
        tours: list[list[int]],
        cost_matrix: list[list[int]]
):
    fitness = []
    tours_count = len(tours)
    for i in range(tours_count):
        tour = [0] + tours[i] + [0]
        fit = 0
        for j in range(len(tour) - 1):
            fit += cost_matrix[tour[j]][tour[j+1]]
        fitness.append(1.0 / fit)

    return fitness

In [151]:
### Children next generation
def cross_over(
        tour_1: list[int],
        tour_2: list[int],
):
    # Init children
    n = len(tour_1)
    a, b = sorted(random.sample(range(n), 2))
    child_1 = [-1] * n
    child_2 = [-1] * n

    # Middle body of children
    child_1[a:(b+1)] = tour_1[a:(b+1)]
    child_2[a:(b+1)] = tour_2[a:(b+1)]

    # Fill the rest of the children
    fill_1 = [city for city in tour_2 if city not in child_1]
    fill_2 = [city for city in tour_1 if city not in child_2]
    ptr1 = ptr2 = 0
    for i in range(n):
        if child_1[i] == -1:
            child_1[i] = fill_1[ptr1]
            ptr1 += 1
        if child_2[i] == -1:
            child_2[i] = fill_2[ptr2]
            ptr2 += 1

    return child_1, child_2

def mutate(
        tour: list[int],
        mutation_rate: float = 0.01
):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(tour)), 2)
        tour[i], tour[j] = tour[j], tour[i]

def create_next_children_generation(
        parents: list[list[int]],
        crossover_func,
        mutate_func,
        mutation_rate: float = 0.01
):
    n = len(parents)
    parents = random.sample(parents, n)
    children = []

    for i in range(0, len(parents)-1, 2):
        p1 = parents[i]
        p2 = parents[i+1]

        c1, c2 = crossover_func(p1, p2)
        mutate_func(c1, mutation_rate)
        mutate_func(c2, mutation_rate)

        children.append(c1)
        children.append(c2)


    return children

In [152]:
### Selection for next generation
def selection_tour_with_range(
        tours: list[list[int]],
        fitnesses: list[int],
        k: int = 3
):
    init_idxs = random.sample(range(len(tours)), k)
    best_idx = max(init_idxs, key = lambda i:  fitnesses[i])
    return tours[best_idx]

def selection_for_next_generation(
        tours: list[list[int]],
        fitnesses: list[int],
        tours_size: int,
        k: int = 3
):
    next_genetation = []
    for i in range(tours_size):
        next_genetation.append(selection_tour_with_range(tours, fitnesses, k))

    return next_genetation

In [153]:
### Best selection
def best_selection(
        tours: list[list[int]],
        fitnesses: list[int]
):
    best_idx = max(range(len(tours)), key = lambda i: fitnesses[i])
    return tours[best_idx], 1.0 / fitnesses[best_idx]

In [154]:
# Main GA TSP
def genetic_algorithm_tsp(
        cost_matrix: list[list[int]],
        tours_size: int,
        max_gen: int,
        mutation_rate: float,
        fitness_func,
        selection_func,
        generation_func,
        crossover_func,
        mutate_func,
        best_selection_func,
        k: int = 3
):
    tours = init_population(len(cost_matrix), tours_size)
    current_gen = 0
    best_tour = best_cost = None
    while True:
        print(f"Generation: {current_gen}")
        # Fitness
        fitnesses = fitness_func(tours, cost_matrix)

        # Find best
        best_tour_local, best_cost_local = best_selection_func(tours, fitnesses)
        print(f"Best tour local: {best_tour_local}, best cost local: {best_cost_local}"),
        if best_cost is None or best_cost_local < best_cost:
            best_tour = best_tour_local
            best_cost = best_cost_local
            print(f"New best tour: {best_tour}, best cost: {best_cost}")

        # Stop condition
        if current_gen >= max_gen:
            break

        # Generate next generation
        children = generation_func(tours, crossover_func, mutate_func, mutation_rate)
        full_current_generation = children + tours

        # Full fitness for selection
        full_fitnesses = fitness_func(full_current_generation, cost_matrix)

        # Selection
        tours = selection_func(full_current_generation, full_fitnesses, tours_size, k)

        current_gen += 1

    return best_tour, best_cost

In [155]:
import math

def generate_euclidean_cost_matrix(n, seed=0):
    random.seed(seed)
    points = [(random.uniform(0,100), random.uniform(0,100)) for _ in range(n)]
    cost = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = points[i][0] - points[j][0]
                dy = points[i][1] - points[j][1]
                cost[i][j] = round(math.hypot(dx, dy), 2)
    return cost

In [156]:
### Test
cost_matrix = generate_euclidean_cost_matrix(20)
print(cost_matrix)

best_tour, best_cost = genetic_algorithm_tsp(
    cost_matrix,
    10,
    2000,
    0.01,
    cal_fitness,
    selection_for_next_generation,
    create_next_children_generation,
    cross_over,
    mutate,
    best_selection,
    k=2
)

print(f"Best tour: {best_tour}"),
print(f"Best cost: {best_cost}")

[[0, 65.47, 48.54, 45.87, 40.71, 26.12, 56.26, 55.55, 23.41, 14.82, 53.5, 9.18, 75.54, 43.58, 21.96, 38.28, 58.58, 80.05, 38.02, 9.19], [65.47, 0, 17.19, 36.59, 32.93, 54.6, 51.59, 19.8, 87.37, 75.21, 48.37, 63.99, 16.64, 35.22, 86.22, 60.9, 56.91, 27.64, 33.02, 57.52], [48.54, 17.19, 0, 29.09, 18.18, 40.92, 41.92, 18.79, 70.19, 58.02, 38.21, 47.76, 30.67, 21.99, 69.06, 46.16, 47.22, 39.27, 20.85, 40.94], [45.87, 36.59, 29.09, 0, 41.57, 23.67, 67.58, 17.37, 69.1, 59.94, 63.74, 39.77, 37.17, 46.57, 67.58, 64.03, 72.5, 37.28, 11.5, 36.71], [40.71, 32.93, 18.18, 41.57, 0, 43.86, 26.01, 36.18, 58.92, 46.14, 22.17, 43.41, 48.27, 5.06, 58.08, 28.19, 30.96, 57.39, 30.52, 35.84], [26.12, 54.6, 40.92, 23.67, 43.86, 0, 67.47, 38.54, 47.81, 40.94, 63.89, 17.95, 59.44, 48.57, 46.19, 56.2, 71.39, 60.82, 21.61, 18.35], [56.26, 51.59, 41.92, 67.58, 26.01, 67.47, 0, 60.71, 66.77, 54.83, 3.84, 62.12, 68.22, 21.03, 66.54, 22.38, 5.37, 78.83, 56.49, 55.0], [55.55, 19.8, 18.79, 17.37, 36.18, 38.54, 60.71,